# Validation reward curves

This notebook reads TensorBoard logs and plots validation reward over training episodes for selected runs.

In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "statistic" else NOTEBOOK_DIR
LOG_ROOT = PROJECT_ROOT / "logs2"

VARIANT = 0
TOTAL_EPISODES = 10_000
TAG = "Reward/validation"

RUNS = [
    {
        "name": "DQN v8.11.26",
        "path": LOG_ROOT / "DQN_v8.11.26_variant_0",
        "color": "tab:blue",
    },
    {
        "name": "DQN v8.12.0",
        "path": LOG_ROOT / "DQN_v8.12.0_variant_0",
        "color": "tab:green",
    },
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Log root: {LOG_ROOT}")

Project root: /Users/zlr/Documents/CodingLab：DRL/CodingLab_DRL_team
Log root: /Users/zlr/Documents/CodingLab：DRL/CodingLab_DRL_team/logs2


In [8]:
def read_scalar_series(run_path: Path, tag: str = TAG, max_step: int | None = TOTAL_EPISODES):
    if not run_path.exists():
        raise FileNotFoundError(f"Missing TensorBoard log directory: {run_path}")

    event_files = list(run_path.glob("events.out.tfevents*"))
    if not event_files:
        raise FileNotFoundError(f"No TensorBoard event file found in: {run_path}")

    accumulator = EventAccumulator(str(run_path), size_guidance={"scalars": 0})
    accumulator.Reload()

    scalar_tags = accumulator.Tags().get("scalars", [])
    if tag not in scalar_tags:
        raise KeyError(f"Tag {tag!r} not found in {run_path}. Available scalar tags: {scalar_tags}")

    series = [(point.step, point.value) for point in accumulator.Scalars(tag)]
    if max_step is not None:
        series = [(step, value) for step, value in series if step <= max_step]

    if not series:
        raise ValueError(f"No scalar points left after filtering {tag!r} to step <= {max_step}")

    return series


curves = []
missing_runs = []

for run in RUNS:
    try:
        series = read_scalar_series(run["path"])
    except Exception as exc:
        missing_runs.append((run["name"], run["path"], exc))
        continue

    best_step, best_value = max(series, key=lambda item: item[1])
    curves.append({**run, "series": series, "best_step": best_step, "best_value": best_value})

for curve in curves:
    print(
        f"{curve['name']}: {len(curve['series'])} points, "
        f"best validation reward = {curve['best_value']:.3f} at episode {curve['best_step']}"
    )

if missing_runs:
    print("\nRuns not plotted:")
    for name, path, exc in missing_runs:
        print(f"- {name}: {path} ({exc})")


Runs not plotted:
- DQN v8.11.26: /Users/zlr/Documents/CodingLab：DRL/CodingLab_DRL_team/logs2/DQN_v8.11.26_variant_0 (Missing TensorBoard log directory: /Users/zlr/Documents/CodingLab：DRL/CodingLab_DRL_team/logs2/DQN_v8.11.26_variant_0)
- DQN v8.12.0: /Users/zlr/Documents/CodingLab：DRL/CodingLab_DRL_team/logs2/DQN_v8.12.0_variant_0 (Missing TensorBoard log directory: /Users/zlr/Documents/CodingLab：DRL/CodingLab_DRL_team/logs2/DQN_v8.12.0_variant_0)


In [9]:
if not curves:
    raise RuntimeError("No curves were loaded. Check RUNS and LOG_ROOT above.")

fig, ax = plt.subplots(figsize=(10, 5.6), dpi=140)

for curve in curves:
    steps = [step for step, _ in curve["series"]]
    rewards = [value for _, value in curve["series"]]
    ax.plot(steps, rewards, label=curve["name"], color=curve["color"], linewidth=2.2)
    ax.scatter(
        [curve["best_step"]],
        [curve["best_value"]],
        color=curve["color"],
        edgecolor="white",
        linewidth=1.2,
        s=70,
        zorder=5,
    )
    ax.vlines(
        curve["best_step"],
        ymin=0,
        ymax=curve["best_value"],
        color=curve["color"],
        linestyle="--",
        linewidth=1.1,
        alpha=0.75,
    )
    ax.hlines(
        curve["best_value"],
        xmin=0,
        xmax=curve["best_step"],
        color=curve["color"],
        linestyle="--",
        linewidth=1.1,
        alpha=0.75,
    )
    ax.annotate(
        f"{curve['best_value']:.2f}",
        xy=(0, curve["best_value"]),
        xytext=(-8, 0),
        textcoords="offset points",
        ha="right",
        va="center",
        color=curve["color"],
        fontsize=9,
        fontweight="bold",
        annotation_clip=False,
    )
    ax.annotate(
        f"ep {curve['best_step']}",
        xy=(curve["best_step"], 0),
        xytext=(0, -18),
        textcoords="offset points",
        ha="center",
        va="top",
        color=curve["color"],
        fontsize=8,
        annotation_clip=False,
    )

ax.text(
    1.02,
    0.5,
    f"variant: {VARIANT}\ntotal episodes: {TOTAL_EPISODES // 1000}k",
    transform=ax.transAxes,
    va="center",
    ha="left",
    fontsize=11,
    bbox={"boxstyle": "round,pad=0.45", "facecolor": "white", "edgecolor": "0.75"},
)

ax.set_title("Validation reward over training episodes", fontsize=14, pad=12)
ax.set_xlabel("Training episode")
ax.set_ylabel("Average validation reward")
ax.set_xlim(0, TOTAL_EPISODES)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.25)
ax.legend(loc="lower right", frameon=True)

fig.tight_layout(rect=(0.08, 0.06, 0.84, 1))

output_path = PROJECT_ROOT / "statistic" / "validation_reward_8_11_26_vs_8_12_0.png"
fig.savefig(output_path, bbox_inches="tight")
print(f"Saved figure to: {output_path}")

plt.show()

RuntimeError: No curves were loaded. Check RUNS and LOG_ROOT above.